# Colormaps

Build perceptually uniform colormaps, look values up in them, check them, simulate color blindness and save them.

In [1]:
import os
import tempfile
from pathlib import Path

import numpy as np

import lcms2
from lcms2 import Colormap

out = Path(os.environ.get("LCMS2_EXAMPLES_OUT") or tempfile.mkdtemp())

`Colormap.from_colors` interpolates between colors in OKLab, so equal steps in data look like equal steps
in color. Colors are hex strings, CSS names, `rgb(...)` strings or numbers.

In [2]:
cmap = Colormap.from_colors(["#0b1a3a", "#f08a24", "#fff6d5"], name="dusk")
print(cmap)
print(cmap.hex()[:3], cmap.colors.shape)

Colormap('dusk', n=256, space='oklab', #0b1a3a -> #fff6d5)
['#0b1a3a', '#0d1b3a', '#0e1c3b'] (256, 3)


`positions` places the colors unevenly; here the middle color sits a quarter of the way in. `space`
changes the interpolation space and `n` the table size.

In [3]:
skewed = Colormap.from_colors(["#0b1a3a", "#f08a24", "#fff6d5"], positions=[0, 0.25, 1], n=8)
print(skewed.hex())
print(Colormap.from_colors(["black", "white"], space="srgb", n=5).hex())

['#0b1a3a', '#875b44', '#f18f32', '#f6a55a', '#f9b97b', '#fcce9a', '#fee2b8', '#fff6d5']
['#000000', '#404040', '#808080', '#bfbfbf', '#ffffff']


Built-ins and generators: `Colormap.get(name)`, `lcms2.colormap.list()`, and `sequential`, `diverging`,
`cyclic` and `qualitative` constructors.

In [4]:
print(lcms2.colormap.list())
maps = [
    Colormap.get("glacier"),
    Colormap.sequential(250),
    Colormap.diverging("#2b5fb4", "#b4352b"),
    Colormap.cyclic(),
    Colormap.qualitative(5),
]
for m in maps:
    print(f"{m.name:15s} monotonic L: {m.is_lightness_monotonic()!s:5s} step spread: {m.uniformity():.3f}")

['ember', 'glacier', 'halo', 'lagoon', 'meadow', 'orchid', 'parade', 'polar', 'sunrise', 'verdant']
glacier         monotonic L: True  step spread: 0.034
sequential-250  monotonic L: True  step spread: 0.034
diverging       monotonic L: False step spread: 0.045
cyclic          monotonic L: True  step spread: 0.000
qualitative-5   monotonic L: True  step spread: 0.000


Calling a colormap looks values up in an array of any shape. Values are scaled by `vmin` and `vmax` (0..1
by default), `bytes=True` returns uint8, `alpha` adds a channel and `nan_color` replaces NaN.

In [5]:
values = np.array([0.0, 0.25, 0.5, 1.0, np.nan])
print(cmap(values, nan_color="#ff00ff").round(3))
print(cmap(values, bytes=True, alpha=0.8)[1])
print(cmap(np.array([[0, 5], [10, 20]]), vmin=0, vmax=10).shape)

[[0.043 0.102 0.227]
 [0.465 0.327 0.268]
 [0.939 0.541 0.145]
 [1.    0.965 0.835]
 [1.    0.    1.   ]]
[119  83  68 204]
(2, 2, 3)


Reverse, resample and slice to derive new maps.

In [6]:
print(cmap.reversed()(0.0).round(3))
print(cmap.resampled(16))
print(cmap[64:192])

[1.    0.965 0.835]
Colormap('dusk', n=16, space='oklab', #0b1a3a -> #fff6d5)
Colormap('dusk', n=128, space='oklab', #775444 -> #fbc186)


`simulate` previews a map as seen with a color vision deficiency; `severity` goes from 0 to 1.

In [7]:
print(cmap.simulate("deuteranopia")(0.5).round(3))
print(cmap.simulate("protanopia", severity=0.5)(0.5).round(3))

[0.757 0.676 0.146]
[0.778 0.581 0.088]


Save to PNG, CSV, JSON or TXT and load a map back. `to_matplotlib()` converts to a matplotlib colormap
and `register()` makes it available by name (matplotlib is optional).

In [8]:
for suffix in ("png", "csv", "json"):
    print(cmap.save(out / f"dusk.{suffix}").name)
print(Colormap.load(out / "dusk.json"))
try:
    mpl = cmap.to_matplotlib()
    print(type(mpl).__name__, mpl.name)
except ImportError:
    print("matplotlib is not installed")

dusk.png
dusk.csv
dusk.json
Colormap('dusk', n=256, space='oklab', #0b1a3a -> #fff6d5)
ListedColormap dusk
